<a href="https://colab.research.google.com/github/mehwishqayyum021-scitech/teleco_chrn-practice_prediction/blob/main/healthmodel_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import pandas as pd
import numpy as np
import os
import kagglehub
import joblib
from joblib import dump, load

from sklearn.experimental import enable_iterative_imputer  # MUST be imported first
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Download and Load Data
path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")
csv_file_path = os.path.join(path, 'heart_disease_uci.csv')
df = pd.read_csv(csv_file_path)

# 2. Target Variable Cleaning
df.replace('?', np.nan, inplace=True)
df['target'] = (df['num'] > 0).astype(int)
df.drop(['num', 'id'], axis=1, inplace=True, errors='ignore')

# 3. Handle Categorical Columns (One-Hot Encoding)
categorical_cols = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
df_encoded = pd.get_dummies(df, columns=[c for c in categorical_cols if c in df.columns], dummy_na=False)

# Ensure all data is float for MICE
X = df_encoded.drop('target', axis=1).astype(float)
y = df_encoded['target']

# Save exact training columns to align the Streamlit app later
joblib.dump(X.columns.tolist(), 'model_columns1.joblib')

# 4. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Apply MICE (IterativeImputer)
print("Applying MICE Imputation...")
mice_imputer = IterativeImputer(random_state=42, max_iter=10)
X_train_imputed = mice_imputer.fit_transform(X_train)
X_test_imputed = mice_imputer.transform(X_test)

# Save the Imputer
joblib.dump(mice_imputer, 'mice_imputer.joblib')

# 6. Train Random Forest Classifier
print("Training Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_imputed, y_train)

# Save the Model
joblib.dump(model, 'heart_model1.joblib')

# 7. Evaluate
y_pred = model.predict(X_test_imputed)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))
print(" Model, Imputer, and Columns saved successfully!")

Using Colab cache for faster access to the 'heart-disease-data' dataset.
Applying MICE Imputation...
Training Model...
Accuracy: 84.78%
              precision    recall  f1-score   support

           0       0.79      0.85      0.82        75
           1       0.89      0.84      0.87       109

    accuracy                           0.85       184
   macro avg       0.84      0.85      0.84       184
weighted avg       0.85      0.85      0.85       184

 Model, Imputer, and Columns saved successfully!


In [35]:
%%writefile heartapp1.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Page setup
st.set_page_config(page_title="Heart Risk CDSS", layout="wide")

# 1. Load the Model, MICE Imputer, and Columns
# Added a check to ensure files exist before loading
try:
    model = joblib.load('heart_model1.joblib')
    mice_imputer = joblib.load('mice_imputer.joblib')
    model_columns = joblib.load('model_columns1.joblib')
except Exception as e:
    st.error(f"Error loading model files: {e}")
    st.stop()

st.title("Heart Disease Risk Predictor")
st.write("Professional Clinical Decision Support Tool (MICE Imputed)")

# 2. Input Fields
col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", 1, 120, 50)
    sex = st.selectbox("Sex", ["Male", "Female"])
    cp = st.selectbox("Chest Pain Type", ["typical angina", "atypical angina", "non-anginal", "asymptomatic"])
    trestbps = st.number_input("Resting BP (mm Hg)", value=120)
    chol = st.number_input("Cholesterol (mg/dl)", value=200)

with col2:
    fbs = st.selectbox("Fasting Blood Sugar > 120", [True, False])
    restecg = st.selectbox("Resting ECG", ["normal", "lv hypertrophy", "st-t wave abnormality"])
    thalch = st.number_input("Max Heart Rate", value=150)
    exang = st.selectbox("Exercise Induced Angina", [True, False])
    oldpeak = st.number_input("ST Depression (Oldpeak)", 0.0, 10.0, 0.0)

# 3. Prediction Logic
if st.button("Analyze Risk"):
    # Create initial dataframe from inputs
    input_df = pd.DataFrame([{
        'age': age, 'sex': sex, 'cp': cp, 'trestbps': trestbps,
        'chol': chol, 'fbs': fbs, 'restecg': restecg,
        'thalch': thalch, 'exang': exang, 'oldpeak': oldpeak
    }])

    # Apply One-Hot Encoding
    input_encoded = pd.get_dummies(input_df)

    # REINDEX: FIXED the variable name here to match the loaded 'model_columns'
    final_input = input_encoded.reindex(columns=model_columns, fill_value=np.nan)

    # Ensure float datatype
    final_input = final_input.astype(float)

    # Apply MICE imputation
    final_input_imputed = mice_imputer.transform(final_input)

    # Predict
    prediction = model.predict(final_input_imputed)
    probability = model.predict_proba(final_input_imputed)[0][1]

    # Display Results
    st.divider()
    if prediction[0] == 1:
        st.error(f"High Risk Detected (Probability: {probability:.2%})")
        st.info("Medical recommendation: Further clinical investigation required.")
    else:
        st.success(f"Low Risk Detected (Probability: {probability:.2%})")

Overwriting heartapp1.py


In [42]:
import os

# 1. Kill any zombie processes
!fuser -k 8501/tcp
!pkill streamlit

# 2. Force-download the specific Linux binary for Bore
print("--- DOWNLOADING BORE BINARY ---")
!curl -Ls https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz | tar -zxf -
!chmod +x ./bore
print("Bore is ready.")

# 3. Launch the App and Bore
# We use get_ipython().system_raw to keep Streamlit running in the background
print("--- LAUNCHING SYSTEM ---")
print("Look for the 'remote port forwarding' line below to find your link!")

from IPython import get_ipython
get_ipython().system_raw('streamlit run heartapp1.py --server.enableCORS false --server.enableXsrfProtection false &')

# This will stay active and show you the 'bore.pub:XXXX' link
!./bore local 8501 --to bore.pub

8501/tcp:            25894
--- DOWNLOADING BORE BINARY ---
Bore is ready.
--- LAUNCHING SYSTEM ---
Look for the 'remote port forwarding' line below to find your link!
2026-03-29T17:26:11.839836Z  INFO bore_cli::client: connected to server remote_port=38313
2026-03-29T17:26:11.839871Z  INFO bore_cli::client: listening at bore.pub:38313


In [ ]:
# 1. Clean up old processes
!fuser -k 8501/tcp
!pkill streamlit

# 2. Install Cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 3. Launch Streamlit in the background
import os
from IPython import get_ipython
get_ipython().system_raw('streamlit run heartapp1.py --server.enableCORS false --server.enableXsrfProtection false &')

# 4. Create the Tunnel (This will generate a link immediately)
print("--- GENERATING SECURE CLOUDFLARE LINK ---")
!cloudflared tunnel --url http://localhost:8501

8501/tcp:            26434
Selecting previously unselected package cloudflared.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
--- GENERATING SECURE CLOUDFLARE LINK ---
2026-03-29T17:35:52Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-29T17:35:52Z INF Reque

In [38]:
# FALLBACK: ONLY use this if Bore fails
!pip install -q streamlit
!npm install -g localtunnel -q
!fuser -k 8501/tcp

import urllib
# Get your password for the tunnel
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"TUNNEL PASSWORD (IP): {ip}")

# Run Streamlit
!streamlit run heartapp1.py & npx localtunnel --port 8501